# Weight Structure Analysis

Analyzes weight matrix spectral properties, parameter utilization, head similarity, and embedding-unembedding relationships.

**References:**
- Martin & Mahoney (2021) "Implicit Self-Regularization in Deep Neural Networks"
- Sharma & Kaplan (2022) "Scaling Laws from the Data Manifold Dimension"

## Why This Matters

Weight structure analysis examines the spectral properties, parameter utilization, and internal organization of weight matrices. The structure of weights constrains what computations are possible, and spectral analysis can reveal low-rank structure, symmetries, and functional specialization.

**Key references:**
- [Elhage et al. (2021) "A Mathematical Framework for Transformer Circuits"](https://transformer-circuits.pub/2021/framework/index.html) — Foundational framework for attention head and residual stream analysis

In [ ]:
import jax
import jax.numpy as jnp
import numpy as np

from irtk import HookedTransformer, HookedTransformerConfig
from irtk.weight_structure import (
    weight_spectral_analysis,
    parameter_utilization,
    head_weight_similarity,
    embedding_weight_relationship,
    layer_weight_norm_profile,
)

cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)
print(f"Model: {cfg.n_layers} layers, {cfg.n_heads} heads")

In [ ]:
# 1. Weight spectral analysis
spec = weight_spectral_analysis(model, top_k=3)
print("Effective ranks:")
for k, v in sorted(spec['effective_ranks'].items()):
    print(f"  {k}: rank={v:.2f}, cond={spec['condition_numbers'].get(k, 'N/A')}")

In [ ]:
# 2. Parameter utilization
util = parameter_utilization(model)
print(f"Total parameters: {util['total_params']:,}")
print(f"Near-zero fraction: {util['near_zero_fraction']:.4f}")
print(f"\nParams by type:")
for t, c in util['param_count_by_type'].items():
    print(f"  {t}: {c:,}")
stats = util['weight_magnitude_stats']
print(f"\nWeight magnitude: mean={stats['mean']:.4f}, std={stats['std']:.4f}, range=[{stats['min']:.6f}, {stats['max']:.4f}]")

In [ ]:
# 3. Head weight similarity
hsim = head_weight_similarity(model)
print(f"Most similar heads: {hsim['most_similar_heads']}")
print(f"Most different heads: {hsim['most_different_heads']}")
for l in range(cfg.n_layers):
    print(f"  Layer {l} within-similarity: {hsim['within_layer_similarity'][l]:.4f}")
print(f"Across-layer similarity: {hsim['across_layer_similarity']:.4f}")

In [ ]:
# 4. Embedding-unembedding relationship
emb = embedding_weight_relationship(model)
print(f"Weight tying score: {emb['weight_tying_score']:.4f}")
print(f"Frobenius distance: {emb['frobenius_distance']:.4f}")
print(f"Rank correlation: {emb['rank_correlation']:.4f}")
print(f"Embed rank: {emb['embed_rank']:.2f}, Unembed rank: {emb['unembed_rank']:.2f}")

In [ ]:
# 5. Weight norm profile
profile = layer_weight_norm_profile(model)
for l in range(cfg.n_layers):
    print(f"Layer {l}: Q={profile['attn_q_norms'][l]:.3f}, K={profile['attn_k_norms'][l]:.3f}, "
          f"V={profile['attn_v_norms'][l]:.3f}, O={profile['attn_o_norms'][l]:.3f}, "
          f"Win={profile['mlp_in_norms'][l]:.3f}, Wout={profile['mlp_out_norms'][l]:.3f}, "
          f"total={profile['total_per_layer'][l]:.3f}")